# Đọc bài viết thành giọng nói bằng VieNeu-TTS

Notebook này đọc một file `.txt` (ví dụ `biet-tin-yeu.txt`) thành file âm thanh MP3, bằng bộ giọng **VieNeu-TTS v3 Turbo** (25 giọng đọc sẵn, giấy phép Apache-2.0). Cùng bộ giọng mà phần mềm Sano dùng.

**Cách dùng**
1. Menu **Runtime → Change runtime type → T4 GPU** (nhanh hơn nhiều; không có GPU vẫn chạy được, chậm hơn).
2. Chạy lần lượt từng ô, từ trên xuống.
3. Ô "Tải file chữ" sẽ hỏi file `.txt`. Ô "Giọng đọc" cho chọn giọng có sẵn hoặc giọng của chính bạn (nhân bản từ một đoạn thu 3 đến 8 giây).

**Lưu ý:** chỉ nhân bản giọng của chính bạn, hoặc giọng người đã cho phép. Mô hình sẽ tự tải về máy chủ Colab ở lần chạy đầu.

## 1. Cài thư viện

In [ ]:
import subprocess, shutil
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print("GPU:", "có" if has_gpu else "không (sẽ chạy bằng CPU, chậm hơn)")
if has_gpu:
    !pip install -q "vieneu[cuda]"
else:
    !pip install -q vieneu

## 2. Tải file chữ lên

In [ ]:
from google.colab import files
up = files.upload()          # chọn file biet-tin-yeu.txt
TXT_NAME = next(iter(up))
text = open(TXT_NAME, encoding="utf-8").read()
paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
print(f"{TXT_NAME}: {len(text.split())} từ, {len(paragraphs)} đoạn")

## 3. Nạp mô hình và xem danh sách giọng

In [ ]:
from vieneu import Vieneu
vieneu = Vieneu()            # mặc định: VieNeu-TTS v3 Turbo, 48 kHz
voices = vieneu.list_preset_voices()
print(f"{len(voices)} giọng có sẵn:")
for label, voice_id in voices:
    print(" -", label)

## 4. Chọn giọng

- **Giọng có sẵn:** đặt `VOICE` bằng đúng tên trong danh sách ở trên (ví dụ `"Hải Đăng"`, `"Mai Anh"`, `"Thái Sơn"`).
- **Giọng của bạn:** đặt `USE_MY_VOICE = True`, rồi ô dưới sẽ hỏi file thu mẫu (`.wav` hoặc `.mp3`, 3 đến 8 giây, nói rõ, ít tạp âm).

In [ ]:
VOICE = "Hải Đăng"
USE_MY_VOICE = False

if USE_MY_VOICE:
    ref = files.upload()
    REF_PATH = next(iter(ref))
    vieneu.add_voice("Giọng của tôi", REF_PATH)   # tự khử tạp âm, cắt tối đa 8 giây
    VOICE = "Giọng của tôi"
print("Giọng đang dùng:", VOICE)

## 5. Nghe thử một đoạn ngắn (nên làm trước khi đọc cả bài)

In [ ]:
from IPython.display import Audio, display
SAMPLE = paragraphs[1] if len(paragraphs) > 1 else paragraphs[0]
audio = vieneu.infer(SAMPLE, voice=VOICE)
vieneu.save(audio, "thu.wav")
display(Audio("thu.wav"))

Nếu nghe thấy chữ nào đọc sai (ví dụ **SEO**, tên nước ngoài), sửa chữ đó trong file `.txt` (viết theo cách đọc, ví dụ **AI** viết hoa thành "Ây ai") rồi chạy lại từ ô 2.

## 6. Đọc cả bài

In [ ]:
import numpy as np, time
SR = 48000
PAUSE = np.zeros(int(SR * 0.6), dtype=np.float32)   # nghỉ 0,6 giây giữa các đoạn

parts, t0 = [], time.time()
for i, p in enumerate(paragraphs, 1):
    audio = vieneu.infer(p, voice=VOICE)
    parts.append(np.asarray(audio, dtype=np.float32))
    parts.append(PAUSE)
    print(f"đoạn {i}/{len(paragraphs)} xong ({time.time()-t0:.0f}s)")

full = np.concatenate(parts)
vieneu.save(full, "bai-viet.wav")
print(f"Xong: {len(full)/SR/60:.1f} phút âm thanh")

## 7. Đổi sang MP3 và tải về

In [ ]:
OUT = TXT_NAME.rsplit(".", 1)[0] + ".mp3"
!ffmpeg -y -loglevel error -i bai-viet.wav -codec:a libmp3lame -qscale:a 4 "{OUT}"
files.download(OUT)